# EMA EPAR Manufacturer Extraction Pipeline (Fixed)

Extracts manufacturer/batch-release sites from EMA European Public Assessment Reports.

**Bug fixes in this version:**
- 🐛 **Fix #1** — Slug-based PDF fallback for JS-rendered EMA pages (main cause of missing PDFs)
- 🐛 **Fix #2** — `MAX_PDF_PAGES` raised from 20 → 60 so section 6.x (manufacturers) is never cut off

## 1. Install Dependencies

## 2. Imports & Constants

In [1]:
from __future__ import annotations

import io, json, re, sys, time, hashlib
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import pandas as pd
import pdfplumber
import requests
from bs4 import BeautifulSoup
from geopy.geocoders import Nominatim
from requests.adapters import HTTPAdapter
from tqdm.notebook import tqdm
from urllib3.util.retry import Retry

In [3]:
# ── EMA endpoints ────────────────────────────────────────────────────────────
EMA_JSON_INDEX_PAGE = (
    "https://www.ema.europa.eu/en/about-us/about-website/"
    "download-website-data-json-data-format"
)
EMA_MEDICINES_JSON_CANDIDATES = [
    "https://www.ema.europa.eu/en/documents/report/"
    "medicines-output-medicines_json-report_en.json",
]

USER_AGENT = (
    "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
    "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0 Safari/537.36 "
    "EMA-EPAR-Research-Pipeline/1.0"
)

REQUEST_TIMEOUT        = 60
SLEEP_BETWEEN_REQUESTS = 5.0
GEOCODE_SLEEP_SECONDS  = 1.2

# FIX #2 — raised from 20 → 60.
# Long SmPCs (combined tablet + oral-solution annexes) can exceed 50 pages before
# reaching section 6.x (batch-release manufacturer). 20 pages was too short.
MAX_PDF_PAGES = 1000

# ── Geography ─────────────────────────────────────────────────────────────────
COUNTRIES = [
    "Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus",
    "Czech Republic", "Czechia", "Denmark", "Estonia", "Finland",
    "France", "Germany", "Greece", "Hungary", "Ireland", "Italy",
    "Latvia", "Lithuania", "Luxembourg", "Malta", "Netherlands",
    "Poland", "Portugal", "Romania", "Slovakia", "Slovenia", "Spain",
    "Sweden", "Norway", "Iceland", "Liechtenstein", "Switzerland",
    "United Kingdom", "UK",
    "India", "China", "United States", "USA", "Canada", "Japan",
    "South Korea", "Republic of Korea", "Singapore", "Israel",
    "Mexico", "Brazil", "Australia",
    # Inverted forms used in some older EMA SmPCs
    "Korea, Republic of",
    "Korea, Democratic People's Republic of",
    "Taiwan, Province of China",
]

MANUFACTURER_SECTION_PATTERNS = [
    r"manufacturer responsible for batch release",
    r"manufacturing site",
    r"manufacturing sites",
    r"name and address of the manufacturer",
    r"name and address of the manufacturers",
    r"batch releas",
    r"site of batch control",
    r"site of batch release",
    r"manufactured by",
]


## 3. Data Model

**New field: `manufacturer_step`**

| Value | Meaning |
|---|---|
| `1` | Manufacturer of the biological active substance (upstream, e.g. CHO cell fermentation) |
| `2` | Manufacturer responsible for batch release (QC, EU-facing release site) |
| `None` | Standard SmPC with no sub-heading distinction (most small-molecule drugs) |

Example — Tepezza:
- Evive Biopharmaceutical Beijing, Ltd → **step 1** (China)
- Catalent Germany Schorndorf GmbH → **step 2** (Germany)

In [5]:
@dataclass
class ManufacturerRecord:
    medicine_name:      Optional[str]
    active_substance:   Optional[str]
    atc_code:           Optional[str]
    medicine_url:       str
    pdf_url:            Optional[str]
    source_block:       str
    manufacturer_name:  Optional[str]
    address:            Optional[str]
    country:            Optional[str]
    manufacturer_step:  Optional[int]   # 1 = active substance maker, 2 = batch release, None = unknown
    latitude:           Optional[float]
    longitude:          Optional[float]

## 4. Utility Helpers

In [7]:
def normalise_space(text: str) -> str:
    # FIX 5: preserve blank lines (paragraph separators between co-listed
    # manufacturers). Only collapse inline whitespace per line.
    lines = text.splitlines()
    normalised = [re.sub(r"[ \t]+", " ", ln).strip() for ln in lines]
    return "\n".join(normalised).strip()

def safe_filename(text: str, max_len: int = 120) -> str:
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text.strip())
    return text[:max_len] if len(text) > max_len else text

def sha1_text(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8", errors="ignore")).hexdigest()

# Map inverted ISO country names → canonical display names
_COUNTRY_ALIASES = {
    "korea, republic of":                "Republic of Korea",
    "korea, democratic people's republic of": "Korea (North)",
    "taiwan, province of china":         "Taiwan",
    "iran, islamic republic of":         "Iran",
    "moldova, republic of":              "Moldova",
    "russian federation":                "Russia",
    "united kingdom":                    "United Kingdom",
    "great britain":                     "United Kingdom",
    "usa":                               "United States",
    "u.s.a.":                            "United States",
    "u.s.":                              "United States",
    # Common OCR artifacts (serif fonts confuse uppercase I with lowercase l)
    "lreland":                           "Ireland",
    "lndia":                             "India",
    "ltaly":                             "Italy",
    "lsrael":                            "Israel",
    "lceland":                           "Iceland",
}

def extract_country(text: str) -> Optional[str]:
    if not text:
        return None
    # Normalise common OCR artifacts before matching
    # (serif fonts often render uppercase I as lowercase l)
    text = re.sub(r"\blreland\b", "Ireland", text)
    text = re.sub(r"\bltaly\b",   "Italy",   text)
    text = re.sub(r"\blndia\b",   "India",   text)
    text_lower = text.lower()
    # Check aliases first (handles inverted forms like "Korea, Republic of")
    for alias, canonical in _COUNTRY_ALIASES.items():
        if alias in text_lower:
            return canonical
    matches = [c for c in COUNTRIES if c.lower() in text_lower]
    if not matches:
        return None
    return sorted(matches, key=len, reverse=True)[0]

def compute_hhi(country_series: pd.Series) -> float:
    shares = country_series.value_counts(normalize=True, dropna=True)
    if shares.empty:
        return float("nan")
    return float((shares ** 2).sum())


## 5. Networking

In [9]:
def build_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=5, read=5, connect=5, backoff_factor=1.5,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET", "HEAD"]),
        respect_retry_after_header=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    session.headers.update({"User-Agent": USER_AGENT})
    return session


def fetch_html(session: requests.Session, url: str, max_attempts: int = 5) -> str:
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            r = session.get(url, timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
            return r.text
        except requests.HTTPError as e:
            last_error = e
            status = getattr(e.response, "status_code", None)
            if status == 429:
                wait = min(15 * attempt, 60)
                print(f"[WARN] 429 for {url}. Sleeping {wait}s …")
                time.sleep(wait)
                continue
            raise
        except Exception as e:
            last_error = e
            wait = min(5 * attempt, 20)
            print(f"[WARN] fetch_html failed for {url}. Retry in {wait}s …")
            time.sleep(wait)
    raise last_error


# Clean browser UA used for EMA PDF downloads — the pipeline UA string
# triggers EMA's WAF and produces 403 on PDF CDN requests.
_EMA_PDF_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/pdf,*/*",
}


def download_binary(session: requests.Session, url: str, max_attempts: int = 5,
                    referer: str = "") -> bytes:
    last_error = None
    # Use clean browser headers for EMA PDF CDN requests
    extra_headers = dict(_EMA_PDF_HEADERS)
    if referer:
        extra_headers["Referer"] = referer
    for attempt in range(1, max_attempts + 1):
        try:
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            r = session.get(url, timeout=REQUEST_TIMEOUT, headers=extra_headers)
            r.raise_for_status()
            content      = r.content
            content_type = r.headers.get("Content-Type", "").lower()
            if len(content) < 1000:
                raise ValueError(f"Downloaded file is too small. Content-Type: {content_type}")
            if not content.startswith(b"%PDF-"):
                raise ValueError(f"Not a real PDF. Content-Type: {content_type}")
            return content
        except requests.HTTPError as e:
            last_error = e
            status = getattr(e.response, "status_code", None)
            if status == 429:
                wait = min(20 * attempt, 90)
                print(f"[WARN] 429 for PDF {url}. Sleeping {wait}s …")
                time.sleep(wait)
                continue
            raise
        except Exception as e:
            last_error = e
            wait = min(5 * attempt, 20)
            print(f"[WARN] download_binary failed for {url}. Retry in {wait}s …")
            time.sleep(wait)
    raise last_error

## 6. FIX #1 — Multi-Candidate Slug-Based PDF Fallback

The EMA website is JS-rendered — `requests.get()` returns no PDF links.

**New in this version:** medicines with multiple formulations have **multiple PDFs**, each with an indexed suffix:
```
keytruda-epar-product-information_en.pdf      <- IV 25 mg/mL
keytruda-epar-product-information-0_en.pdf    <- SC 395 mg
keytruda-epar-product-information-1_en.pdf    <- SC 790 mg
```
`try_constructed_pdf_url()` now tries **4 candidates** (primary + `-0`, `-1`, `-2`) and returns all that respond HTTP 200.

In [11]:
def construct_pdf_candidates(medicine_url: str) -> List[str]:
    """
    Generate all plausible product-information PDF URLs for a given medicine slug.

    EMA uses a stable naming convention but multi-formulation products get
    indexed variants:
        {slug}-epar-product-information_en.pdf       <- primary
        {slug}-epar-product-information-0_en.pdf     <- 2nd formulation
        {slug}-epar-product-information-1_en.pdf     <- 3rd formulation
        {slug}-epar-product-information-2_en.pdf     <- 4th formulation

    Example - Keytruda (pembrolizumab) has 3 PDFs:
        keytruda-epar-product-information_en.pdf       (IV 25 mg/mL)
        keytruda-epar-product-information-0_en.pdf     (SC 395 mg)
        keytruda-epar-product-information-1_en.pdf     (SC 790 mg)
    """
    slug = medicine_url.rstrip("/").split("/")[-1].lower()
    base = (
        "https://www.ema.europa.eu/en/documents/product-information/"
        f"{slug}-epar-product-information"
    )
    return [
        f"{base}_en.pdf",
        f"{base}-0_en.pdf",
        f"{base}-1_en.pdf",
        f"{base}-2_en.pdf",
    ]


def try_constructed_pdf_url(
    session: requests.Session,
    medicine_url: str,
) -> List[str]:
    """
    Fallback for JS-rendered EMA pages that expose no <a href> PDF links.
    Tries the primary slug URL AND indexed variants (-0, -1, -2) to capture
    all formulations.

    Strategy: HEAD first (fast, low bandwidth). If HEAD returns 403/405
    (EMA's CDN blocks HEAD on PDFs — e.g. Cancidas), fall back to a streaming
    GET that reads only the first 5 bytes to check for the %PDF- magic number.
    A 404 on either method means the URL does not exist.
    Returns list of all confirmed PDF URLs (may be 0-4 entries).
    """
    candidates = construct_pdf_candidates(medicine_url)
    confirmed: List[str] = []

    print(
        f"[INFO] No PDF links found in HTML (JS-rendered page). "
        f"Trying {len(candidates)} constructed URL(s) ..."
    )

    def _url_exists(url: str) -> bool:
        """Return True if the URL serves a real PDF (HEAD or streaming GET)."""
        try:
            time.sleep(1)
            r = session.head(url, timeout=REQUEST_TIMEOUT)
            if r.status_code == 200:
                return True
            if r.status_code == 404:
                return False
            # HEAD blocked (403, 405, etc.) — fall back to streaming GET
            print(f"[INFO] HEAD returned {r.status_code}, trying GET for: {url}")
        except Exception as e:
            print(f"[WARN] HEAD failed ({e}), trying GET for: {url}")

        try:
            time.sleep(1)
            r = session.get(url, timeout=REQUEST_TIMEOUT, stream=True)
            if r.status_code == 404:
                r.close()
                return False
            if r.status_code == 200:
                # Read just the first 5 bytes to verify it's a real PDF
                header = next(r.iter_content(5), b"")
                r.close()
                return header.startswith(b"%PDF-")
        except Exception as e:
            print(f"[WARN] GET also failed for {url}: {e}")
        return False

    for candidate in candidates:
        if _url_exists(candidate):
            print(f"[INFO] Confirmed: {candidate}")
            confirmed.append(candidate)

    if not confirmed:
        print(f"[WARN] No constructed PDF URLs confirmed for {medicine_url}")
    return confirmed


## 7. EMA Data Download

In [13]:
def discover_medicines_json_url(session: requests.Session) -> str:
    html = fetch_html(session, EMA_JSON_INDEX_PAGE)
    soup = BeautifulSoup(html, "lxml")
    candidates: List[str] = []
    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        text = a.get_text(" ", strip=True).lower()
        if not href:
            continue
        if href.startswith("/"):
            href = "https://www.ema.europa.eu" + href
        href_lower = href.lower()
        if href_lower.endswith(".json") and (
            "medicine" in href_lower or "medicines" in href_lower
            or "medicine pages json data file" in text or "medicine pages" in text
        ):
            candidates.append(href)
    for c in candidates:
        cl = c.lower()
        if "medicines-output" in cl or "medicine" in cl:
            return c
    if candidates:
        return candidates[0]
    raise ValueError("Could not discover medicines JSON URL from EMA JSON index page.")


def _records_from_json_payload(data) -> List[dict]:
    if isinstance(data, list):
        return [x for x in data if isinstance(x, dict)]
    if isinstance(data, dict):
        list_candidates = [
            v for v in data.values()
            if isinstance(v, list) and len(v) > 0 and isinstance(v[0], dict)
        ]
        if list_candidates:
            return list_candidates[0]
        raise ValueError(f"Could not find a record list. Top-level keys: {list(data.keys())}")
    raise ValueError(f"Unexpected EMA JSON structure: {type(data)}")


def download_ema_table(session: requests.Session) -> pd.DataFrame:
    last_error = None
    for url in EMA_MEDICINES_JSON_CANDIDATES:
        try:
            r = session.get(url, timeout=REQUEST_TIMEOUT)
            r.raise_for_status()
            data    = r.json()
            records = _records_from_json_payload(data)
            if not records:
                raise ValueError("No records found in candidate JSON URL.")
            df = pd.DataFrame(records)
            df.columns = [str(c).strip() for c in df.columns]
            print(f"[INFO] Using EMA medicines JSON: {url}")
            return df
        except Exception as e:
            last_error = e
    discovered_url = discover_medicines_json_url(session)
    r = session.get(discovered_url, timeout=REQUEST_TIMEOUT)
    r.raise_for_status()
    data    = r.json()
    records = _records_from_json_payload(data)
    if not records:
        raise ValueError(f"EMA JSON returned no records. Last error: {last_error}")
    df = pd.DataFrame(records)
    df.columns = [str(c).strip() for c in df.columns]
    print(f"[INFO] Using discovered EMA medicines JSON: {discovered_url}")
    return df

## 8. Medicine Preparation & Filtering

In [15]:
def prepare_medicines_df(df: pd.DataFrame) -> pd.DataFrame:
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    def col(*names: str) -> Optional[str]:
        for name in names:
            if name.lower() in lower_map:
                return lower_map[name.lower()]
        for existing_lower, original in lower_map.items():
            for name in names:
                if name.lower() in existing_lower:
                    return original
        return None

    name_col      = col("name_of_medicine", "medicine", "medicine name", "name")
    substance_col = col("active_substance", "international_non_proprietary_name_common_name", "active substance")
    atc_col       = col("atc_code_human", "atc code", "atc_code")
    status_col    = col("medicine_status", "authorisation_status", "status")
    url_col       = col("medicine_url", "url", "link")
    mah_col       = col("marketing_authorisation_developer_applicant_holder",
                        "marketing authorisation developer applicant holder")

    missing = []
    if name_col is None: missing.append("name_of_medicine / medicine")
    if url_col  is None: missing.append("medicine_url / url")
    if missing:
        raise ValueError(f"Required EMA JSON columns missing: {missing}. Available: {list(df.columns)}")

    out = pd.DataFrame({
        "medicine_name":                  df[name_col].astype(str).str.strip(),
        "active_substance":               df[substance_col].astype(str).str.strip() if substance_col else None,
        "atc_code":                       df[atc_col].astype(str).str.strip()       if atc_col       else None,
        "authorisation_status":           df[status_col].astype(str).str.strip()    if status_col    else None,
        "medicine_url":                   df[url_col].astype(str).str.strip(),
        "marketing_authorisation_holder": df[mah_col].astype(str).str.strip()       if mah_col       else None,
    })
    out = out[out["medicine_name"].notna()].copy()
    out = out[out["medicine_url"].str.contains(r"ema\.europa\.eu", case=False, na=False)].copy()
    out = out.drop_duplicates(subset=["medicine_url"]).reset_index(drop=True)
    return out


def filter_medicines(
    df: pd.DataFrame,
    atc_prefix:       Optional[str]  = None,
    active_substance: Optional[str]  = None,
    url_file:         Optional[Path] = None,
    authorised_only:  bool           = True,
) -> pd.DataFrame:
    out = df.copy()
    if authorised_only and "authorisation_status" in out.columns:
        status = out["authorisation_status"].fillna("").str.lower()
        out = out[status.str.contains("authorised", na=False) | status.str.contains("authorized", na=False)].copy()
    if atc_prefix:
        out = out[out["atc_code"].fillna("").str.upper().str.startswith(atc_prefix.strip().upper())].copy()
    if active_substance:
        query = active_substance.strip().lower()
        out = out[out["active_substance"].fillna("").str.lower().str.contains(re.escape(query), na=False)].copy()
    if url_file:
        urls = {l.strip() for l in url_file.read_text(encoding="utf-8").splitlines() if l.strip()}
        out = out[out["medicine_url"].isin(urls)].copy()
    return out.reset_index(drop=True)

## 9. PDF Link Extraction & Text Parsing

**Adaptive page reading strategy:**
```
Short PDFs (<= 120 pages) -> read ALL pages
Long PDFs  (>  120 pages) -> first 30 + last 80 pages
```
Keytruda's combined IV+SC PDF is 200+ pages. Annex II (manufacturer) sits ~page 180 — within the last-80 window. The old `TAIL_PAGES=40` missed it for very long documents.

The extractor prints total pages and strategy chosen per PDF, making truncation issues visible.

### 9b. PDF Link Scraping Functions

These two functions scrape `<a href>` tags from the medicine page HTML. For JS-rendered EMA pages they return an empty list — `process_medicine` then falls back to `try_constructed_pdf_url()` (Section 6).

In [18]:
def extract_pdf_links_from_medicine_page(html: str) -> List[str]:
    """
    Scrape product-information PDF links from the medicine page HTML.
    Returns an empty list for JS-rendered pages — caller must use the slug fallback.
    """
    soup  = BeautifulSoup(html, "lxml")
    links: List[str] = []
    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        text = (a.get_text(" ", strip=True) or "").lower()
        if not href or href.startswith("#"):
            continue
        if href.startswith("/"):
            href = "https://www.ema.europa.eu" + href
        href_lower = href.lower()
        if href_lower.endswith("_en.pdf") and "product-information" in href_lower:
            links.append(href)
            continue
        if "product information" in text and href_lower.endswith("_en.pdf"):
            links.append(href)
    return list(dict.fromkeys(links))


def resolve_document_to_pdf(session: requests.Session, url: str) -> List[str]:
    """Follow a document page link and extract product-information PDF URLs from it."""
    url = (url or "").strip()
    if not url or url.startswith("#"):
        return []
    url_lower = url.lower()
    if url_lower.endswith("_en.pdf") and "product-information" in url_lower:
        return [url]
    if ".pdf" in url_lower and not url_lower.endswith("_en.pdf"):
        return []
    try:
        html = fetch_html(session, url)
    except Exception:
        return []
    soup = BeautifulSoup(html, "lxml")
    pdfs = []
    for a in soup.find_all("a", href=True):
        href = (a.get("href") or "").strip()
        if not href or href.startswith("#"):
            continue
        if href.startswith("/"):
            href = "https://www.ema.europa.eu" + href
        href_lower = href.lower()
        if href_lower.endswith("_en.pdf") and "product-information" in href_lower:
            pdfs.append(href)
    return list(dict.fromkeys(pdfs))


In [19]:
# ── Page reading strategy ────────────────────────────────────────────────────
#
# EMA combined product-information PDFs are structured as:
#   Annex I   -- SmPC  (can be 150-200+ pages for complex biologics)
#   Annex II  -- MAH + Manufacturer responsible for batch release  <- we need this
#   Annex III -- Labelling (a few pages)
#
# Keytruda example: 200+ pages total, Annex II around page 180.
# Old TAIL_PAGES=40 only read pages 161-200+ -- caught it for some docs but not all.
#
# NEW STRATEGY:
#   Short PDFs (<= MAX_FULL_READ_PAGES=120): read ALL pages -- safest.
#   Long PDFs  (> 120 pages):  head (30) + tail (80 pages).
#   TAIL_PAGES=80 ensures Annex II is always within the window even for 280-page docs.

MAX_FULL_READ_PAGES = 500
HEAD_PAGES          = 30
TAIL_PAGES          = 200


def _get_total_pages(pdf_bytes: bytes) -> int:
    try:
        import pdfplumber, io
        with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
            return len(pdf.pages)
    except Exception:
        pass
    try:
        from pypdf import PdfReader
        import io
        return len(PdfReader(io.BytesIO(pdf_bytes)).pages)
    except Exception:
        pass
    return 9999


def _build_page_indices(total: int) -> List[int]:
    if total <= MAX_FULL_READ_PAGES:
        return list(range(total))
    head = list(range(min(HEAD_PAGES, total)))
    tail = list(range(max(0, total - TAIL_PAGES), total))
    seen, merged = set(), []
    for idx in head + tail:
        if idx not in seen:
            seen.add(idx)
            merged.append(idx)
    return merged


def extract_pdf_text_head_tail(pdf_bytes: bytes, **kwargs) -> str:
    """
    Smart extractor: reads all pages for short PDFs, head+tail for long ones.
    Falls back through pdfplumber -> pypdf -> PyMuPDF.
    """
    total   = _get_total_pages(pdf_bytes)
    indices = _build_page_indices(total)
    strategy = "all pages" if total <= MAX_FULL_READ_PAGES else f"head({HEAD_PAGES})+tail({TAIL_PAGES})"
    print(f"[INFO] PDF: {total} total pages, reading {len(indices)} ({strategy})")

    def _pdfplumber(data, idxs):
        import pdfplumber, io
        texts = []
        with pdfplumber.open(io.BytesIO(data)) as pdf:
            for i in idxs:
                if i >= len(pdf.pages): break
                try:   txt = pdf.pages[i].extract_text() or ""
                except: txt = ""
                if txt: texts.append(txt)
        return "\n".join(texts)

    def _pypdf(data, idxs):
        from pypdf import PdfReader
        import io
        reader = PdfReader(io.BytesIO(data))
        texts = []
        for i in idxs:
            if i >= len(reader.pages): break
            txt = reader.pages[i].extract_text() or ""
            if txt: texts.append(txt)
        return "\n".join(texts)

    def _pymupdf(data, idxs):
        import fitz, io
        doc = fitz.open(stream=data, filetype="pdf")
        texts = []
        for i in idxs:
            if i >= len(doc): break
            txt = doc[i].get_text("text") or ""
            if txt: texts.append(txt)
        return "\n".join(texts)

    for extractor in [_pdfplumber, _pypdf, _pymupdf]:
        try:
            text = extractor(pdf_bytes, indices)
            if text.strip():
                return text
        except Exception:
            continue
    raise ValueError("All PDF text extractors failed")


# Compatibility aliases
def extract_pdf_text(pdf_bytes: bytes, max_pages: int = MAX_FULL_READ_PAGES) -> str:
    return extract_pdf_text_head_tail(pdf_bytes)

def extract_pdf_text_with_fallback(pdf_bytes: bytes, **kwargs) -> str:
    return extract_pdf_text_head_tail(pdf_bytes)


## 10. Manufacturer Block Extraction & Parsing

**Bug fixes in this version:**

**Fix 1 (original)** — Biological active substance heading added.

**Fix 2 (original)** — Sub-heading split for step-1/step-2 biologics.

**Fix 3 (original)** — Expanded COMPANY_RE.

**Fix 4 (NEW) — MAH/local-representative contamination** 
Blocks containing 'marketing authorisation holder', 'local representative', or
the country-by-country contact table were being mistakenly captured as manufacturer
records. Three-layer fix:
- `MAH_SECTION_STOP_RE` stops the section collector at MAH lines
- `_is_mah_block()` rejects whole blocks that are clearly MAH/rep tables
- `SECTION_END_PATTERNS` extended with MAH stop markers


In [21]:
# ── Constants ─────────────────────────────────────────────────────────────────
DOSAGE_FORMS = {
    "tablets", "tablet", "oral solution", "oral solutions",
    "film-coated tablets", "film-coated tablet",
    "capsules", "capsule", "hard capsules", "soft capsules",
    "solution for injection", "solution for infusion",
    "concentrate for solution for infusion",
    "concentrate for solution", "powder for solution",
    "granules", "suppositories", "patches", "patch",
    "cream", "gel", "ointment", "eye drops", "ear drops",
    "nasal spray", "inhaler", "syrup", "suspension", "emulsion",
    "implant", "pessaries", "drops", "spray", "foam",
    "prolonged-release tablets", "modified-release tablets",
    "dispersible tablets", "effervescent tablets",
    "chewable tablets", "orodispersible tablets",
    "pre-filled syringe", "pre-filled syringes",
    "autoinjector", "autoinjectors",
    "solution for subcutaneous injection",
    "powder for concentrate for solution for infusion",
    "powder for concentrate",
}

DOCUMENT_STRUCTURE_PATTERNS = [
    r"^\s*annex\s+[ivxlc0-9]+\b",
    r"^\s*summary of product characteristics",
    r"^\s*\d+\.\s*name of the medicinal product",
    r"^\s*package leaflet",
    r"^\s*labelling",
    r"^\s*b\.\s+conditions or restrictions",
    r"^\s*c\.\s+other conditions",
]

ADDRESS_STOP_PATTERNS = [
    r"^\s*b[\.\)]\s+conditions or restrictions",
    r"^\s*c[\.\)]\s+other conditions",
    r"^\s*d[\.\)]\s+conditions or restrictions with regard",
    r"^\s*annex\s+[ivxlc0-9]+\b",
    r"^\s*the printed package leaflet",
    r"^\s*medicinal product subject to",
    r"^\s*periodic safety update",
    r"^\s*risk management plan",
    # r"^\\s*\\d+\\s*$",  # REMOVED: falsely matches ZIP codes
    # FIX 4: stop before MAH / local-rep sections
    r"marketing authorisation holder",
    r"for any information about this medicinal? product",
    r"^\s*local representative",
]

# FIX 1: Added "biological active substance" variant to catch Tepezza-style headings
SECTION_START_PATTERNS = [
    r"manufacturer.*responsible.*batch release",
    r"manufacturer\(s\).*responsible.*batch release",
    r"manufacturers.*responsible.*batch release",
    r"manufacturer.*biological active substance",
    r"name and address of the manufacturer",
    r"name and address of the manufacturers",
    r"manufacturing site[s]?$",
    r"site of batch release",
    r"site of batch control",
    r"^manufactured by",  # anchor: only match as a standalone heading, not mid-sentence
]

SECTION_END_PATTERNS = [
    r"^\s*b[\.\)]\s+conditions or restrictions",
    r"^\s*c[\.\)]\s+other conditions",
    r"^\s*d[\.\)]\s+conditions or restrictions with regard",
    r"^\s*annex\s+(i|ii|iii|iv|v)\b",
    r"^\s*the printed package leaflet",
    r"^\s*labelling",
    r"^\s*medicinal product subject to",
    r"^\s*periodic safety update",
    r"^\s*risk management plan",
    # FIX 4: stop before MAH / local-rep sections
    r"marketing authorisation holder",
    r"for any information about this medicinal? product",
]

# FIX 2: These are TOP-LEVEL headings to strip (A./B. section letters)
TOP_HEADING_PATTERNS = [
    r"manufacturer.*responsible.*batch release",
    r"manufacturer\(s\).*responsible.*batch release",
    r"manufacturers.*responsible.*batch release",
    r"manufacturer.*biological active substance",
    r"manufacturers of the biological active substance",  # combined heading variant
    r"manufacturers responsible for batch release",       # combined heading variant
    r"manufacturing site", r"manufacturing sites",
    r"site of batch control", r"site of batch release",
    r"manufactured by", r"batch release site",
]

# FIX 2: "Name and address of..." lines are SUB-HEADINGS that delimit different
# groups of manufacturers. They should NOT be stripped — they are used as split points.
SUB_HEADING_PATTERNS = [
    r"name and address of the manufacturer",
    r"name and address of the manufacturers",
]

# FIX 3: Expanded COMPANY_RE with therapeutics, dac, biotech, and many more
# FIX 5b: word boundaries prevent short tokens (inc, ag, bv, nv…) matching
# inside place names (Incheon, Nagyvárad, Brno, Novara…).
COMPANY_RE = re.compile(
    r"(?<!\w)(gmbh|llc|kft\.|k\.f\.t\.|a\.s\.|spol\.\s*s\s*r\.o\.|s\.r\.o\.|limited|ltd|inc\.?|s\.a\.u\.|s\.a\.|s\.p\.a\.|b\.v\.|n\.v\.|bv|nv|ag|oy|ab|sas|sae|dac|"
    r"pharma|laboratories|laboratoire|farmaceutici|pharmaceuticals|biologics|"
    r"biologicals|bioscience|biosciences|biotech|biotechnology|therapeutics|"
    r"healthcare|medac|teva|pfizer|novartis|roche|sanofi|merck|"
    r"janssen|sandoz|abbvie|amgen|bayer|astrazeneca|gsk|lilly|viiv|ucb|"
    r"manufacturing|chugai|nextpharma|aesica|vetter|lonza|boehringer|"
    r"fresenius|hospira|baxter|strides|apotex|mylan|accord|actavis|"
    r"glaxo|wellcome|smithkline|wockhardt|cipla|hikma|zentiva|krka|"
    r"gedeon|richter|takeda|fujifilm|daiichi|otsuka|agc|allergan|"
    r"catalent|patheon|recipharm|famar|almac|piramal|siegfried|rentschler|"
    r"evonik|thermo fisher|samsung|celltrion|msd|eli lilly|"
    r"horizon|genmab|regeneron|biogen|alexion|shire|idec|"
    r"nordic|nordic bioscience|nordic pharma|genzyme|sanofi-aventis|winthrop|rensselaer|dupont|lonza|wuxi biologics|wuxi apptec|charles river|covance|novatek|novasep|halix|abzena|rentschler|polymun|rentschler|siegfried|pliva|teva|delpharm|synthelabo|fareva|famar|cenexi|euro-pharma|elaiapharm|haupt pharma|salutas|lannett|amneal|aurobindo|zydus|sun pharma|teva|noven|human biologics)(?!\w)",
    re.I,
)

# FIX 4: Pattern that identifies MAH / local-representative blocks — these must
# NEVER be treated as manufacturer records.
MAH_SECTION_STOP_RE = re.compile(
    r"marketing authorisation holder"
    r"|for any information about this medicinal? product"
    r"|please contact the local representative"
    r"|local representative of the marketing authorisation",
    re.I,
)

def _is_mah_block(block: str) -> bool:
    """Return True if this block is a MAH / local-rep section, not a manufacturer block."""
    return bool(MAH_SECTION_STOP_RE.search(block))

STANDALONE_OR_RE = re.compile(r"^\s*or\s*$", re.I)
INLINE_OR_RE     = re.compile(r"(?<!\w)\s+or\s+(?!\w)", re.I)


def _is_document_structure(block: str) -> bool:
    first = block.strip().splitlines()[0].lower() if block.strip() else ""
    return any(re.search(p, first) for p in DOCUMENT_STRUCTURE_PATTERNS)


def _is_top_heading(ln: str) -> bool:
    "Top-level headings to strip — but NEVER sub-headings (they carry step info)."
    if _is_sub_heading(ln):  # FIX 5c: sub-headings must survive to classify steps
        return False
    cleaned = re.sub(r"^[a-z0-9]+[\.\)\-]\s*", "", ln.lower().strip())
    cleaned = re.sub(r"\(s\)", "", cleaned).strip(" :-")
    return any(re.search(p, cleaned) for p in TOP_HEADING_PATTERNS)


def _is_sub_heading(ln: str) -> bool:
    "Sub-headings used as split points ('Name and address of the manufacturer...')"
    cleaned = ln.lower().strip()
    return any(re.search(p, cleaned) for p in SUB_HEADING_PATTERNS)


# Regex for dash-prefixed product-specific form labels, e.g.
# "- Plavix 75 mg film-coated tablets", "- Keytruda 25 mg/ml concentrate"
# Matches product-specific form labels, both with dash ("- Plavix 75 mg...")
# and without (after strip(" -\t") in the parser: "Plavix 75 mg...")
_DASH_FORM_RE = re.compile(
    r"^-?\s*\S.*\b(tablet|capsule|solution|infusion|injection|"
    r"suspension|powder|granule|implant|patch|spray|cream|gel|drop|film)\b",
    re.I,
)
# Stricter version: also requires Brand + dose pattern (e.g. "Plavix 75 mg")
_PRODUCT_FORM_RE = re.compile(
    r"^[A-Z][a-zA-Z]+\s+\d.*\b(tablet|capsule|solution|infusion|injection|"
    r"suspension|powder|granule|implant|patch|spray|cream|gel|drop|film)\b",
    re.I,
)

def _is_dosage_form(ln: str) -> bool:
    s = ln.strip().lower()
    if s in DOSAGE_FORMS:
        return True
    stripped = ln.strip()
    # "- Plavix 75 mg film-coated tablets" (with dash)
    if _DASH_FORM_RE.match(stripped):
        return True
    # "Plavix 75 mg film-coated tablets" (dash already stripped by parser)
    if _PRODUCT_FORM_RE.match(stripped):
        return True
    return False


def _is_address_stop(ln: str) -> bool:
    ln_lower = ln.strip().lower()
    return any(re.search(p, ln_lower) for p in ADDRESS_STOP_PATTERNS)


def _is_section_end(ln: str) -> bool:
    ln_lower = ln.strip().lower()
    return any(re.search(p, ln_lower) for p in SECTION_END_PATTERNS)


def _truncate_at_stop(lines: list) -> list:
    for i, ln in enumerate(lines):
        if _is_address_stop(ln):
            return lines[:i]
    return lines


def _find_company(lines: list) -> tuple:
    for i, ln in enumerate(lines[:8]):
        if not _is_dosage_form(ln) and not re.match(r"^[0-9]", ln):
            if COMPANY_RE.search(ln):
                return ln, i
    for i, ln in enumerate(lines):
        if not _is_dosage_form(ln) and not re.match(r"^[0-9B\-]", ln):
            return ln, i
    return lines[0], 0


# Matches only unambiguous legal suffixes (not brand names) for inline address splitting
_LEGAL_SUFFIX_RE = re.compile(
    r"(?<!\w)(gmbh|limited|ltd|inc\.?|s\.a\.u\.|s\.a\.|s\.p\.a\.|b\.v\.|n\.v\.|bv|nv)(?!\w)",
    re.I,
)


def _split_inline_address(line: str) -> tuple:
    """
    FIX: Handle older EMA SmPCs where the full address is on one line, e.g.:
      'Sanofi-Aventis Deutschland GmbH, Industriepark Hoechst, 65926 Frankfurt am Main, Germany'
    Uses the LAST legal-suffix token (GmbH, S.A., B.V. …) as the company name boundary
    so that compound names like 'Sanofi-Aventis Deutschland GmbH' are kept intact.
    """
    best_end = -1
    for m in _LEGAL_SUFFIX_RE.finditer(line):
        best_end = m.end()
    if best_end < 0:
        return line, []
    sep = best_end
    while sep < len(line) and line[sep] in " ,":
        sep += 1
    company = line[:best_end].strip(" ,")
    rest    = line[sep:].strip()
    if not rest:
        return company, []
    return company, [p.strip() for p in rest.split(",") if p.strip()]


def _parse_single_side(lines: list, fallback_block: str = "") -> tuple:
    lines = [ln for ln in lines if ln.strip()]
    if not lines:
        return None, None, None
    lines = _truncate_at_stop(lines)
    if not lines:
        return None, None, None
    company, idx = _find_company(lines)
    addr_lines   = [ln for ln in lines[idx + 1:] if not _is_dosage_form(ln)]

    # FIX: detect inline comma-separated address (one-line format in older SmPCs)
    if not addr_lines and "," in company:
        company, addr_lines = _split_inline_address(company)

    address  = "\n".join(addr_lines).strip() or None
    # Prefer last address line — country is always last in EMA addresses.
    _addr_tail = addr_lines[-1] if addr_lines else ""
    country = (
        extract_country(_addr_tail)
        or extract_country("\n".join(addr_lines[-3:]))
        or extract_country("\n".join(addr_lines))
        or extract_country(company)
        or extract_country(fallback_block)
    )
    return company, address, country
def _split_at_or(section_lines: list) -> list:
    # Case A: standalone 'or' line
    if any(STANDALONE_OR_RE.match(ln) for ln in section_lines):
        groups, current = [], []
        for ln in section_lines:
            if STANDALONE_OR_RE.match(ln):
                if current:
                    groups.append(current)
                current = []
            else:
                current.append(ln)
        if current:
            groups.append(current)
        return groups or [section_lines]

    # Case B: inline 'or'
    if any(INLINE_OR_RE.search(ln) for ln in section_lines):
        left_lines, right_lines = [], []
        for ln in section_lines:
            m = INLINE_OR_RE.search(ln)
            if m:
                left_lines.append(ln[:m.start()].strip())
                right_lines.append(ln[m.end():].strip())
            else:
                words = ln.split()
                mid   = len(words) // 2
                left_lines.append(" ".join(words[:mid]).strip())
                right_lines.append(" ".join(words[mid:]).strip())
        groups = []
        for g in [left_lines, right_lines]:
            g = [l for l in g if l]
            if g:
                groups.append(g)
        return groups or [section_lines]

    return [section_lines]


def _split_at_company_boundaries(lines: list) -> list:
    "Split at each new company name line within a group."
    if not lines:
        return [lines]
    groups, current = [], []
    for ln in lines:
        if (COMPANY_RE.search(ln)
                and current
                and not _is_dosage_form(ln)
                and len(current) > 1):
            groups.append(current)
            current = [ln]
        else:
            current.append(ln)
    if current:
        groups.append(current)
    return groups if len(groups) > 1 else [lines]



def _split_at_blank_lines(lines: list) -> list:
    """FIX 5: split at blank lines — the natural paragraph separator between
    co-listed manufacturers in EMA SmPCs.
    """
    groups: list = []
    current: list = []
    for ln in lines:
        if not ln.strip():
            if current:
                groups.append(current)
            current = []
        else:
            current.append(ln)
    if current:
        groups.append(current)
    return groups if len(groups) > 1 else [lines]

def extract_manufacturer_sections(text: str) -> list:
    """
    Section-aware extractor: finds each manufacturer section heading and
    collects all lines until the next major section boundary.

    Blank lines within a section are PRESERVED as empty strings so that
    _split_companies() can use them as company delimiters.
    FIX 4: also stops at MAH / local-representative lines.
    """
    lines = text.splitlines()   # do NOT strip here — blank lines must survive
    sections = []
    i = 0
    while i < len(lines):
        line       = lines[i].strip()
        line_lower = line.lower()
        if any(re.search(pat, line_lower) for pat in SECTION_START_PATTERNS):
            section_lines = [line]
            i += 1
            while i < len(lines):
                current       = lines[i]          # keep raw — strip only for tests
                current_strip = current.strip()
                # Stop at section-end or MAH markers
                if _is_section_end(current_strip) or MAH_SECTION_STOP_RE.search(current_strip):
                    break
                # Preserve blank lines as empty strings (company separators)
                section_lines.append(current_strip)
                i += 1
            block = "\n".join(section_lines).strip()
            if block:
                # Use normalise_space which preserves blank lines (Fix 5)
                sections.append(normalise_space(block))
        else:
            i += 1
    return sections

def split_into_candidate_blocks(text: str) -> list:
    sections   = extract_manufacturer_sections(text)
    raw_blocks = [normalise_space(b) for b in re.split(r"\n{2,}", text) if b.strip()]
    all_blocks = sections + raw_blocks
    seen, out  = set(), []
    for b in all_blocks:
        h = sha1_text(b)
        if h not in seen:
            seen.add(h)
            out.append(b)
    return out


def keep_relevant_blocks(blocks: list) -> list:
    out = []
    for block in blocks:
        # FIX 4: reject MAH / local-representative blocks unconditionally
        if _is_mah_block(block):
            continue
        if _is_document_structure(block):
            continue
        bl = block.lower()
        if any(re.search(pat, bl) for pat in MANUFACTURER_SECTION_PATTERNS):
            out.append(block)
            continue
        # Also match the biological active substance variant
        if re.search(r"manufacturer.*biological active substance", bl):
            out.append(block)
            continue
        # Fallback: block has a country AND looks like a company address.
        # FIX 4: guard against rep-table blocks that also match this condition.
        # FIX 6+7: fallback requires a real legal entity suffix (COMPANY_RE),
        # not just the word "manufacturer" anywhere in clinical text.
        # Also reject blocks whose first non-empty line is a page number,
        # bullet, symbol, or short fragment — those are garbled PDF output.
        first_non_empty = next(
            (ln.strip() for ln in block.splitlines() if ln.strip()), ""
        )
        if re.match(r"^[\d•\-–—β≥≤×·]+$", first_non_empty):
            continue  # page number, bullet, or symbol-only line — not a manufacturer block
        if len(first_non_empty) < 4:
            continue  # too short to be a real company/heading line
        if extract_country(block) and COMPANY_RE.search(bl):
            out.append(block)
    seen, dedup = set(), []
    for b in out:
        h = sha1_text(b)
        if h not in seen:
            seen.add(h)
            dedup.append(b)
    return dedup


# ── Sub-heading classification ───────────────────────────────────────────────
# These patterns identify which type of manufacturer each sub-section describes.
# Step 1 = manufacturer of the biological active substance (upstream)
# Step 2 = manufacturer responsible for batch release (QC/release, EU-facing)
# None   = not specified / standard small-molecule SmPC with no sub-headings

STEP1_SUBHEADING_RE = re.compile(
    r"name and address of the manufacturer.*biological active substance",
    re.I,
)
STEP2_SUBHEADING_RE = re.compile(
    r"name and address of the manufacturer.*responsible for batch release"
    r"|name and address of the manufacturer[s]?\s*$",
    re.I,
)

def _classify_subheading(ln: str) -> Optional[int]:
    "Return 1, 2, or None based on which sub-heading type this line is."
    if STEP1_SUBHEADING_RE.search(ln):
        return 1
    if STEP2_SUBHEADING_RE.search(ln):
        return 2
    return None


def parse_manufacturer_block_multi(
    block: str,
) -> List[Tuple[Optional[str], Optional[str], Optional[str], Optional[int]]]:
    """
    Parse a manufacturer block into (name, address, country, step) tuples.

    Algorithm
    ---------
    1. Strip top-level headings (e.g. "A. MANUFACTURER RESPONSIBLE FOR BATCH RELEASE").
       Sub-heading lines ("Name and address of...") are NOT stripped — they carry step info.
    2. Split at sub-headings to assign step labels:
         step 1  -> biological active substance manufacturer
         step 2  -> manufacturer responsible for batch release
         step None -> no sub-heading (standard small-molecule SmPC)
    3. Within each step-group, split individual companies by (priority order):
         a) blank line              — primary separator in EMA SmPCs
         b) standalone "or" / "and" line
         c) inline "or" / "and" within a line
         d) COMPANY_RE boundary    — fallback when PDF has no blank lines
    """
    lines = [ln.strip(" -\t") for ln in block.splitlines()]  # keep blank lines!
    if not lines:
        return []

    # 1. Strip top-level headings — but never strip sub-headings
    lines = [ln for ln in lines if not _is_top_heading(ln)]
    if not lines:
        return []

    # 2. Split at sub-headings, record step per group
    sub_groups: List[Tuple[List[str], Optional[int]]] = []
    current:      List[str]    = []
    current_step: Optional[int] = None
    has_sub_headings = False

    for ln in lines:
        if _is_sub_heading(ln):
            has_sub_headings = True
            if current:
                sub_groups.append((current, current_step))
            current      = []
            current_step = _classify_subheading(ln)
        else:
            current.append(ln)
    if current:
        sub_groups.append((current, current_step))

    if not sub_groups:
        sub_groups = [(lines, None)]

    if has_sub_headings:
        assigned, counter = [], 1
        for grp, step in sub_groups:
            if step is None:
                step = counter
            counter = (step or 0) + 1
            assigned.append((grp, step))
        sub_groups = assigned

    # 3. Split each step-group into individual company records
    STANDALONE_SPLIT_RE = re.compile(r"^\s*(or|and)\s*$", re.I)
    INLINE_SPLIT_RE     = re.compile(r"\s+or\s+|\s+and\s+", re.I)  # FIX: \s+ guards suffice; dropped (?!\w) which blocked "or Aesica"

    def _company_re_split(lines: list) -> List[list]:
        """Split a flat lines-list at COMPANY_RE boundaries (fallback)."""
        grps: list = []
        cur: list  = []
        for ln in lines:
            if COMPANY_RE.search(ln) and cur and not _is_dosage_form(ln) and len(cur) > 1:
                grps.append(cur)
                cur = [ln]
            else:
                cur.append(ln)
        if cur:
            grps.append(cur)
        return grps if len(grps) > 1 else [lines]

    def _split_companies(grp_lines: list) -> List[list]:
        """Split lines into per-company sublists."""
        # a) blank-line split (primary)
        groups: list = []
        current: list = []
        for ln in grp_lines:
            if not ln.strip():
                if current:
                    groups.append(current)
                current = []
            else:
                current.append(ln)
        if current:
            groups.append(current)
        if len(groups) > 1:
            # FIX: also apply COMPANY_RE boundary split within each blank-group
            # so that multiple companies inside one form-section are split too
            result: list = []
            for g in groups:
                result.extend(_company_re_split(g))
            return result

        non_blank = [ln for ln in grp_lines if ln.strip()]

        # b) standalone or/and line
        if any(STANDALONE_SPLIT_RE.match(ln) for ln in non_blank):
            groups, current = [], []
            for ln in non_blank:
                if STANDALONE_SPLIT_RE.match(ln):
                    if current:
                        groups.append(current)
                    current = []
                else:
                    current.append(ln)
            if current:
                groups.append(current)
            return groups or [non_blank]

        # c) inline or/and
        if any(INLINE_SPLIT_RE.search(ln) for ln in non_blank):
            left, right = [], []
            for ln in non_blank:
                m = INLINE_SPLIT_RE.search(ln)
                if m:
                    left.append(ln[:m.start()].strip())
                    right.append(ln[m.end():].strip())
                else:
                    words = ln.split()
                    mid   = len(words) // 2
                    left.append(" ".join(words[:mid]))
                    right.append(" ".join(words[mid:]))
            result = [[l for l in s if l] for s in [left, right]]
            return [r for r in result if r] or [non_blank]

        # d) COMPANY_RE boundary (fallback — no blank lines in this PDF)
        grps: list = []
        cur: list  = []
        for ln in non_blank:
            if COMPANY_RE.search(ln) and cur and not _is_dosage_form(ln) and len(cur) > 1:
                grps.append(cur)
                cur = [ln]
            else:
                cur.append(ln)
        if cur:
            grps.append(cur)
        return grps if len(grps) > 1 else [non_blank]

    results: List[Tuple] = []

    for group, step in sub_groups:
        # Replace dosage-form lines (including dash-prefixed product labels)
        # with blank separators so _split_companies splits at them
        cleaned = ["" if _is_dosage_form(ln) else ln for ln in group]

        # FIX: pass group lines (not full block) as fallback_block so
        # country detection stays within this step's address group
        _group_fallback = "\n".join(ln for ln in cleaned if ln.strip())
        for company_lines in _split_companies(cleaned):
            name, addr, country = _parse_single_side(company_lines, _group_fallback)
            if name:
                results.append((name, addr, country, step))

    # Deduplicate
    seen, deduped = set(), []
    for name, addr, country, step in results:
        addr_key = (addr or "").strip().splitlines()[0].lower() if addr else ""
        key = (
            (name    or "").strip().lower(),
            (country or "").strip().lower(),
            addr_key,
            step,
        )
        if key not in seen:
            seen.add(key)
            deduped.append((name, addr, country, step))

    return deduped


def parse_manufacturer_block(block: str) -> tuple:
    "Legacy single-result wrapper (returns without step)."
    results = parse_manufacturer_block_multi(block)
    return results[0][:3] if results else (None, None, extract_country(block))


## 11. Geocoding

In [23]:
def build_geocoder() -> Nominatim:
    return Nominatim(user_agent="ema_epar_manufacturer_pipeline")


def geocode_address(
    geocoder: Nominatim,
    query: str,
    cache: Dict[str, Dict[str, Optional[float]]],
) -> Tuple[Optional[float], Optional[float]]:
    if not query:
        return None, None
    if query in cache:
        cached = cache[query]
        return cached.get("lat"), cached.get("lon")
    time.sleep(GEOCODE_SLEEP_SECONDS)
    try:
        loc = geocoder.geocode(query, timeout=30)
    except Exception:
        loc = None
    lat, lon     = (loc.latitude, loc.longitude) if loc else (None, None)
    cache[query] = {"lat": lat, "lon": lon}
    return lat, lon

## 12. Core Medicine Processor

This is where both fixes are applied:
- **Fix #1** — after HTML scraping, if `pdf_links` is empty, call `try_constructed_pdf_url()`
- **Fix #2** — `max_pages=MAX_PDF_PAGES` (60) passed to all text-extraction calls

In [25]:
def process_medicine(
    session: requests.Session,
    medicine_row: pd.Series,
    pdf_cache_dir: Path,
    html_cache_dir: Path,
    use_geocoding: bool,
    geocoder: Optional[Nominatim],
    geocode_cache: Dict[str, Dict[str, Optional[float]]],
) -> List[ManufacturerRecord]:
    medicine_url     = medicine_row["medicine_url"]
    medicine_name    = medicine_row.get("medicine_name")
    active_substance = medicine_row.get("active_substance")
    atc_code         = medicine_row.get("atc_code")

    print(f"[INFO] Processing: {medicine_name} | {medicine_url}")

    def looks_like_real_pdf(data: bytes) -> bool:
        return len(data) > 1000 and data.startswith(b"%PDF-")

    def load_or_download_pdf(pdf_path: Path, pdf_url: str) -> bytes:
        if pdf_path.exists():
            pdf_bytes = pdf_path.read_bytes()
            if looks_like_real_pdf(pdf_bytes):
                return pdf_bytes
            print(f"[WARN] Cached file is not a real PDF, re-downloading: {pdf_url}")
            pdf_path.unlink(missing_ok=True)
        pdf_bytes = download_binary(session, pdf_url, referer=medicine_url)
        if not looks_like_real_pdf(pdf_bytes):
            raise ValueError("Downloaded file is not a valid PDF")
        pdf_path.write_bytes(pdf_bytes)
        return pdf_bytes

    # ── Fetch / cache medicine page HTML ──────────────────────────────────────
    html_name = safe_filename(medicine_url.split("/")[-1] or sha1_text(medicine_url)) + ".html"
    html_path = html_cache_dir / html_name
    try:
        if html_path.exists():
            html = html_path.read_text(encoding="utf-8")
        else:
            html = fetch_html(session, medicine_url)
            html_path.write_text(html, encoding="utf-8")
    except Exception as e:
        print(f"[WARN] Could not fetch medicine page: {medicine_url} :: {e}")
        return []

    # ── Collect PDF links via HTML scraping ───────────────────────────────────
    raw_links = extract_pdf_links_from_medicine_page(html)
    pdf_links: List[str] = []
    for link in raw_links:
        resolved = resolve_document_to_pdf(session, link)
        if resolved:
            pdf_links.extend(resolved)

    # ── FIX #1 — slug-based fallback for JS-rendered EMA pages ───────────────
    # EMA now renders document links via React; BeautifulSoup finds no <a href>
    # tags pointing at PDFs. If scraping returned nothing, derive the URL from
    # the medicine slug and confirm it exists with a HEAD request.
    if not pdf_links:
        pdf_links = try_constructed_pdf_url(session, medicine_url)

    pdf_links_dedup = list(dict.fromkeys(pdf_links))
    print(f"[INFO] {medicine_name}: {len(pdf_links_dedup)} English product-information PDF(s) found")

    records: List[ManufacturerRecord] = []

    for pdf_url in pdf_links_dedup:
        pdf_url_lower = pdf_url.lower()
        if not pdf_url_lower.endswith("_en.pdf"):
            continue
        if "product-information" not in pdf_url_lower:
            continue

        try:
            print(f"[INFO] Parsing PDF: {pdf_url}")
            pdf_name = safe_filename(pdf_url.split("/")[-1] or sha1_text(pdf_url))
            if not pdf_name.lower().endswith(".pdf"):
                pdf_name += ".pdf"
            pdf_path  = pdf_cache_dir / pdf_name
            pdf_bytes = load_or_download_pdf(pdf_path, pdf_url)

            try:
                # FIX #2 — pass MAX_PDF_PAGES (60) so section 6.x is never cut off
                text = extract_pdf_text_with_fallback(pdf_bytes)  # head+tail strategy
            except Exception as e:
                msg = str(e)
                if "No /Root object!" in msg or "Is this really a PDF?" in msg:
                    print(f"[WARN] Corrupted cached PDF, re-downloading: {pdf_url}")
                    pdf_path.unlink(missing_ok=True)
                    pdf_bytes = download_binary(session, pdf_url, referer=medicine_url)
                    if not looks_like_real_pdf(pdf_bytes):
                        raise ValueError("Re-downloaded file is not a valid PDF")
                    pdf_path.write_bytes(pdf_bytes)
                    text = extract_pdf_text(pdf_bytes)  # head+tail strategy
                else:
                    raise

            if not text.strip():
                print(f"[WARN] No text extracted from PDF: {pdf_url}")
                continue

            candidate_blocks = split_into_candidate_blocks(text)
            relevant_blocks  = keep_relevant_blocks(candidate_blocks)
            print(f"[INFO] {medicine_name}: {len(relevant_blocks)} relevant manufacturer block(s) found")

            for block in relevant_blocks:
                # parse_manufacturer_block_multi returns (name, address, country, step)
                # step=1 → active substance manufacturer
                # step=2 → batch release manufacturer
                # step=None → no sub-heading distinction
                parsed_entries = parse_manufacturer_block_multi(block)
                if not parsed_entries:
                    continue  # FIX: skip heading-only blocks — no parseable manufacturer

                for entry in parsed_entries:
                    manufacturer_name = entry[0]
                    address           = entry[1]
                    country           = entry[2]
                    mfr_step          = entry[3] if len(entry) > 3 else None

                    lat = lon = None
                    if use_geocoding and geocoder is not None:
                        geo_query = ", ".join(
                            x for x in [address, country] if x and str(x).strip()
                        ).strip(", ")
                        if geo_query:
                            lat, lon = geocode_address(geocoder, geo_query, geocode_cache)

                    records.append(ManufacturerRecord(
                        medicine_name    = medicine_name    if pd.notna(medicine_name)    else None,
                        active_substance = active_substance if pd.notna(active_substance) else None,
                        atc_code         = atc_code         if pd.notna(atc_code)         else None,
                        medicine_url=medicine_url, pdf_url=pdf_url, source_block=block,
                        manufacturer_name=manufacturer_name, address=address, country=country,
                        manufacturer_step=mfr_step,
                        latitude=lat, longitude=lon,
                    ))

        except Exception as e:
            print(f"[WARN] Could not process PDF: {pdf_url} :: {e}")

    # De-duplicate
    seen_keys: set = set()
    deduped: List[ManufacturerRecord] = []
    for r in records:
        key = (
            r.medicine_url, r.pdf_url,
            (r.manufacturer_name or "").strip().lower(),
            (r.address          or "").strip().lower(),
            (r.country          or "").strip().lower(),
        )
        if key not in seen_keys:
            seen_keys.add(key)
            deduped.append(r)

    print(f"[INFO] {medicine_name}: {len(deduped)} extracted manufacturer record(s)")
    return deduped


## 13. HHI Concentration Analysis

In [27]:
def build_hhi_tables(manufacturer_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    usable = manufacturer_df.dropna(subset=["country"]).copy()
    if usable.empty:
        return pd.DataFrame(), pd.DataFrame()

    def _make_rows(groupby_col, group_key):
        rows = []
        for val, grp in usable.groupby(groupby_col, dropna=True):
            if pd.isna(val):
                continue
            vc = grp["country"].value_counts()
            rows.append({
                group_key:           val,
                "n_records":         len(grp),
                "n_countries":       grp["country"].nunique(dropna=True),
                "top_country":       vc.idxmax(),
                "top_country_share": vc.max() / vc.sum(),
                "hhi":               compute_hhi(grp["country"]),
            })
        return pd.DataFrame(rows).sort_values(["hhi", "n_records"], ascending=[False, False])

    return _make_rows("active_substance", "active_substance"), _make_rows("atc_code", "atc_code")

## 14. Pipeline Configuration

Set your parameters here, then run Sections 15–16 in order.

In [29]:
# ── User-configurable parameters ─────────────────────────────────────────────
OUTPUT_DIR       = Path("ema_output_steps")   # Where all outputs are saved
ATC_PREFIX       = None                  # e.g. "J01" for antibiotics
ACTIVE_SUBSTANCE = None                  # e.g. "maraviroc"
URL_FILE         = None                  # Path to a .txt file with one URL per line
MAX_MEDICINES    = None                    # Set to None to process everything (slow!)
AUTHORISED_ONLY  = True
ENABLE_GEOCODING = False

# ── Directory setup ───────────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cache_dir      = OUTPUT_DIR / "cache"; cache_dir.mkdir(exist_ok=True)
pdf_cache_dir  = cache_dir  / "pdfs";  pdf_cache_dir.mkdir(exist_ok=True)
html_cache_dir = cache_dir  / "html";  html_cache_dir.mkdir(exist_ok=True)

print(f"Output directory : {OUTPUT_DIR.resolve()}")
print(f"Max PDF pages    : {MAX_PDF_PAGES}  (Fix #2)")

Output directory : /Users/elmadervic/Documents/Github/MedicationsSupply/Scripts/EMA/ema_output_steps
Max PDF pages    : 1000  (Fix #2)


## 15. Download & Filter

In [31]:
session = build_session()

# Step 1 — download EMA medicine list
print("[1/6] Downloading EMA medicines JSON …")
raw_df = download_ema_table(session)
print("Columns:", raw_df.columns.tolist())
raw_df.to_csv(OUTPUT_DIR / "ema_medicines_raw.csv", index=False)
raw_df.head(3)

[1/6] Downloading EMA medicines JSON …
[INFO] Using EMA medicines JSON: https://www.ema.europa.eu/en/documents/report/medicines-output-medicines_json-report_en.json
Columns: ['category', 'name_of_medicine', 'ema_product_number', 'medicine_status', 'opinion_status', 'latest_procedure_affecting_product_information', 'international_non_proprietary_name_common_name', 'active_substance', 'therapeutic_area_mesh', 'species_veterinary', 'patient_safety', 'atc_code_human', 'atcvet_code_veterinary', 'pharmacotherapeutic_group_human', 'pharmacotherapeutic_group_veterinary', 'therapeutic_indication', 'accelerated_assessment', 'additional_monitoring', 'advanced_therapy', 'biosimilar', 'conditional_approval', 'exceptional_circumstances', 'generic', 'orphan_medicine', 'prime_priority_medicine', 'marketing_authorisation_developer_applicant_holder', 'european_commission_decision_date', 'start_of_rolling_review_date', 'start_of_evaluation_date', 'opinion_adopted_date', 'withdrawal_of_application_date', 

,category,name_of_medicine,ema_product_number,medicine_status,opinion_status,latest_procedure_affecting_product_information,international_non_proprietary_name_common_name,active_substance,therapeutic_area_mesh,species_veterinary,...,opinion_adopted_date,withdrawal_of_application_date,marketing_authorisation_date,refusal_of_marketing_authorisation_date,withdrawal_expiry_revocation_lapse_of_marketing_authorisation_date,suspension_of_marketing_authorisation_date,revision_number,first_published_date,last_updated_date,medicine_url
0,Human,Eydenzelt,EMEA/H/C/005899,Authorised,,VR/0000348371,aflibercept,aflibercept,Wet Macular Degeneration;Macular Edema;Diabete...,,...,12/12/2024,,12/02/2025,,,,4,27/03/2025,16/06/2026,https://www.ema.europa.eu/en/medicines/human/E...
1,Human,Kefdensis,EMEA/H/C/006490,Authorised,,VR/0000343436,denosumab,denosumab,"Bone Resorption;Osteoporosis;Osteoporosis, Pos...",,...,18/09/2025,,17/11/2025,,,,2,19/09/2025,16/06/2026,https://www.ema.europa.eu/en/medicines/human/E...
2,Human,Kadcyla,EMEA/H/C/002389,Authorised,,VR/0000348702,trastuzumab emtansine,trastuzumab emtansine,Breast Neoplasms,,...,19/09/2013,,15/11/2013,,,,18,19/12/2013,16/06/2026,https://www.ema.europa.eu/en/medicines/human/E...


In [32]:
# Step 2 — prepare & filter
print("[2/6] Preparing and filtering medicines …")
meds_df = prepare_medicines_df(raw_df)
meds_df = filter_medicines(
    meds_df,
    atc_prefix       = ATC_PREFIX,
    active_substance = ACTIVE_SUBSTANCE,
    url_file         = URL_FILE,
    authorised_only  = AUTHORISED_ONLY,
)
if MAX_MEDICINES:
    meds_df = meds_df.head(MAX_MEDICINES).copy()

meds_df.to_csv(OUTPUT_DIR / "ema_medicines_filtered.csv", index=False)
print(f"Selected medicines: {len(meds_df)}")
meds_df.head()

[2/6] Preparing and filtering medicines …
Selected medicines: 1858


,medicine_name,active_substance,atc_code,authorisation_status,medicine_url,marketing_authorisation_holder
0,Eydenzelt,aflibercept,S01LA05,Authorised,https://www.ema.europa.eu/en/medicines/human/E...,Celltrion Healthcare Hungary Kft.
1,Kefdensis,denosumab,M05BX04,Authorised,https://www.ema.europa.eu/en/medicines/human/E...,Stada Arzneimittel AG
2,Kadcyla,trastuzumab emtansine,L01FD03,Authorised,https://www.ema.europa.eu/en/medicines/human/E...,Roche Registration GmbH
3,Nucala,Mepolizumab,R03DX09,Authorised,https://www.ema.europa.eu/en/medicines/human/E...,GlaxoSmithKline Trading Services Limited
4,Ziihera,zanidatamab,L01FD07,Authorised,https://www.ema.europa.eu/en/medicines/human/E...,Jazz Pharmaceuticals Ireland Limited


## 16. Run the Pipeline

In [34]:
# Step 3 — process medicines & extract manufacturers
print("[3/6] Processing medicine pages and PDFs …")

geocoder           = build_geocoder() if ENABLE_GEOCODING else None
geocode_cache_path = cache_dir / "geocode_cache.json"
geocode_cache      = (
    json.loads(geocode_cache_path.read_text(encoding="utf-8"))
    if geocode_cache_path.exists() else {}
)

incremental_cols = [
    "medicine_name", "active_substance", "atc_code", "medicine_url", "pdf_url",
    "source_block", "manufacturer_name", "address", "country", "manufacturer_step", "latitude", "longitude",
]
incremental_path = OUTPUT_DIR / "manufacturers_extracted_incremental.csv"
if not incremental_path.exists():
    pd.DataFrame(columns=incremental_cols).to_csv(incremental_path, index=False)

try:
    processed_urls = set(pd.read_csv(incremental_path)["medicine_url"].dropna().astype(str))
except Exception:
    processed_urls = set()

all_records, failures = [], []

for _, row in tqdm(meds_df.iterrows(), total=len(meds_df)):
    medicine_url  = row["medicine_url"]
    medicine_name = row.get("medicine_name", medicine_url)
    if medicine_url in processed_urls:
        print(f"[INFO] Skipping already processed: {medicine_name}")
        continue
    try:
        records = process_medicine(
            session        = session,
            medicine_row   = row,
            pdf_cache_dir  = pdf_cache_dir,
            html_cache_dir = html_cache_dir,
            use_geocoding  = ENABLE_GEOCODING,
            geocoder       = geocoder,
            geocode_cache  = geocode_cache,
        )
        all_records.extend(records)
        if records:
            step_labels = {1: "bio.active", 2: "batch release", None: ""}
            for r in records:
                step_tag = (
                    f" [step {r.manufacturer_step}: {step_labels.get(r.manufacturer_step, '')}]"
                    if r.manufacturer_step else ""
                )
                name  = r.manufacturer_name or '(unknown)'
                ctry  = r.country or '?'
                print(f'  -> {name} | {ctry}{step_tag}')
        if records:
            for r in records:
                step_label = {1: "step-1 (active substance)", 2: "step-2 (batch release)"}.get(r.manufacturer_step, "step-? (unknown)")
                print(f"  [EXTRACTED] {r.manufacturer_name} | {r.country} | {step_label}")
        batch = pd.DataFrame([asdict(r) for r in records]) if records else pd.DataFrame([{
            "medicine_name": row.get("medicine_name"), "active_substance": row.get("active_substance"),
            "atc_code": row.get("atc_code"), "medicine_url": medicine_url,
            **{k: None for k in ["pdf_url","source_block","manufacturer_name","address","country","latitude","longitude"]}
        }])
        # Replace old rows for this medicine (keeps CSV clean on re-process)
        if medicine_url in set(pd.read_csv(incremental_path)["medicine_url"].dropna().astype(str)):
            _existing = pd.read_csv(incremental_path)
            _existing = _existing[_existing["medicine_url"] != medicine_url]
            pd.concat([_existing, batch], ignore_index=True).to_csv(incremental_path, index=False)
        else:
            batch.to_csv(incremental_path, mode="a", header=False, index=False)
        processed_urls.add(medicine_url)
        if ENABLE_GEOCODING:
            geocode_cache_path.write_text(
                json.dumps(geocode_cache, ensure_ascii=False, indent=2), encoding="utf-8"
            )
    except Exception as e:
        failures.append({"medicine_url": medicine_url, "medicine_name": medicine_name, "error": str(e)})
        print(f"[WARN] Failed: {medicine_name} :: {e}")

print(f"Done. {len(all_records)} records extracted, {len(failures)} failures.")


[3/6] Processing medicine pages and PDFs …


  0%|          | 0/1858 [00:00<?, ?it/s]

[INFO] Skipping already processed: Eydenzelt
[INFO] Skipping already processed: Kefdensis
[INFO] Skipping already processed: Kadcyla
[INFO] Skipping already processed: Nucala
[INFO] Skipping already processed: Ziihera
[INFO] Skipping already processed: Startvac
[INFO] Skipping already processed: Lonquex
[INFO] Skipping already processed: Opzelura
[INFO] Skipping already processed: Palsonify
[INFO] Skipping already processed: Entyvio
[INFO] Skipping already processed: Rimmyrah
[INFO] Skipping already processed: Rezzayo
[INFO] Skipping already processed: Zolgensma
[INFO] Skipping already processed: Grasustek
[INFO] Skipping already processed: Tecovirimat SIGA
[INFO] Skipping already processed: Sitagliptin / Metformin hydrochloride Mylan
[INFO] Skipping already processed: Tadalafil Mylan
[INFO] Skipping already processed: Tuyory
[INFO] Skipping already processed: Zutectra
[INFO] Skipping already processed: Emtricitabine/Tenofovir disoproxil Mylan
[INFO] Skipping already processed: Dimethy

In [35]:
# FIX 4 — Mark medicines for re-processing (does NOT delete existing rows)
#
# Add medicine names to REPROCESS_MEDICINES to force them to be re-processed
# on the next Step 3 run. Their existing rows are KEPT in the CSV until Step 3
# successfully produces new ones — so you never lose data mid-run.
#
# How it works:
#   - Removes the medicine URL from processed_urls → Step 3 will re-run it
#   - Does NOT touch the incremental CSV rows → old data stays until overwritten
#   - Step 3 deduplicates by (medicine_url, pdf_url, manufacturer_name, address,
#     country) so re-running produces clean output without duplicates

REPROCESS_MEDICINES = [
    # medicines whose PDF was previously missed due to WAF blocking pipeline UA
    "oncaspar",
    "xromi",
    "siklos",
    "darzalex",
    "advate",
    # medicines whose PDF was previously missed due to HEAD-request blocking
    "cancidas",
    "norvir",
    "kaletra",
    "rinvoq",
    "venclyxto",
    "ozurdex",
    "humira",
    "aquipta",
    # ilumira: fix lreland OCR country and drop empty records
    "ilumira",
    # clopidogrel: fix dosage-form separator + delpharm in COMPANY_RE
    "clopidogrel",
    # jakavi: fix _company_re_split (extracted before fix)
    "jakavi",
    # lyrica: fix or/blank-line split
    "lyrica",
]

if REPROCESS_MEDICINES and incremental_path.exists():
    inc_df = pd.read_csv(incremental_path)
    pattern = "|".join(re.escape(m) for m in REPROCESS_MEDICINES)

    # Find URLs of medicines to re-process
    urls_to_reprocess = set(
        inc_df.loc[
            inc_df["medicine_name"].fillna("").str.contains(pattern, case=False, regex=True),
            "medicine_url"
        ].dropna().astype(str)
    )

    # Only remove from processed_urls set — do NOT touch the CSV
    before = len(processed_urls)
    processed_urls -= urls_to_reprocess
    n_queued = len(urls_to_reprocess)

    if n_queued:
        print(f"[INFO] {n_queued} medicine(s) queued for re-processing on next Step 3 run:")
        for url in sorted(urls_to_reprocess):
            name = inc_df.loc[inc_df["medicine_url"] == url, "medicine_name"].iloc[0] if url in inc_df["medicine_url"].values else url
            print(f"  {name}")
        print(f"[INFO] Existing rows kept in CSV — will be replaced when Step 3 runs.")
    else:
        print("[INFO] No matching medicines found in incremental CSV.")
else:
    print("[INFO] No re-processing requested.")


[INFO] 28 medicine(s) queued for re-processing on next Step 3 run:
  Advate
  Aquipta
  Cancidas (previously Caspofungin MSD)
  Clopidogrel / Acetylsalicylic acid Viatris (previously Clopidogrel / Acetylsalicylic acid Mylan)
  Clopidogrel/Acetylsalicylic acid Zentiva (previously DuoCover)
  Clopidogrel BGR (previously Zylagren)
  Clopidogrel HCS
  Clopidogrel Krka
  Clopidogrel Krka d.d. (previously Zopya)
  Clopidogrel ratiopharm
  Clopidogrel TAD
  Clopidogrel Taw Pharma (previously Clopidogrel Mylan)
  Clopidogrel Teva (hydrogen sulphate)
  Clopidogrel Viatris (previously Clopidogrel Taw Pharma)
  Clopidogrel Zentiva (previously Clopidogrel Winthrop)
  Darzalex
  Humira
  Ilumira
  Jakavi
  Kaletra
  Lyrica
  Norvir
  Oncaspar
  Ozurdex
  Rinvoq
  Siklos
  Venclyxto
  Xromi
[INFO] Existing rows kept in CSV — will be replaced when Step 3 runs.


## 15b. Re-derive Country from Address (skip Step 3 re-run)

Run this cell instead of re-running Step 3 when only the country extraction logic has changed.
Reads the existing `manufacturers_extracted_incremental.csv` and rewrites the `country` column
using the current `extract_country` function — no HTTP requests made.


In [37]:
# ── Re-derive country from address across ALL collected records ───────────────
# Run this cell whenever the country-extraction logic changes.
# Also drops rows where manufacturer_name is clearly a sentence fragment,
# clinical text, bullet, or symbol (bad parse from mid-sentence triggers).
# NO network calls — all data comes from the already-stored address column.

print("[FIX] Re-deriving country and dropping bad rows for ALL records …")

_inc_path   = OUTPUT_DIR / "manufacturers_extracted_incremental.csv"
_final_path = OUTPUT_DIR / "manufacturers_extracted.csv"

if not _inc_path.exists():
    print("[WARN] No incremental CSV found — run Step 3 first.")
else:
    _df = pd.read_csv(_inc_path)
    n_before = len(_df)

    # ── Step 1: drop rows where manufacturer_name is clearly not a company name ──
    _BAD_PHRASE_RE = re.compile(
        r"should\s+be\s+used|be\s+used\s+with|to\s+refill|to\s+rinse|"
        r"to\s+flush|to\s+fill|to\s+dissolve|"
        r"^the\s+following|^during\s+|^in\s+patients|^patients\s+|"
        r"^cystoid|^adverse\s+event|^these\s+|^this\s+medicine|"
        r"were\s+reported|has\s+been\s+(un)?commonly\s+reported",
        re.I,
    )
    _LOWERCASE_START_RE = re.compile(r"^\s*[a-z]")  # case-sensitive — no re.I
    _BAD_SYMBOL_RE = re.compile(
        r"^[\d\u2022\-\u2013\u2014\u03b2\u2265\u2264\u00d7\u00b7\s]+$"
        r"|^\(|[\u2265\u2264\u03b2\u2022]",
        re.U,
    )

    def _is_bad_name(n: str) -> bool:
        return (
            bool(_BAD_PHRASE_RE.search(n))
            or bool(_LOWERCASE_START_RE.match(n))
            or bool(_BAD_SYMBOL_RE.search(n))
            or len(n.strip()) < 4
        )

    _bad_mask = _df["manufacturer_name"].fillna("").apply(lambda n: _is_bad_name(str(n)))
    n_dropped = int(_bad_mask.sum())
    if n_dropped:
        print(f"[FIX] Dropping {n_dropped} bad row(s) (sentence-fragment / symbol manufacturer names):")
        for _, r in _df[_bad_mask].iterrows():
            med = str(r.get("medicine_name", "?"))[:20]
            mfr = str(r["manufacturer_name"])[:60]
            print(f"  {med:20}  {mfr!r}")
    _df = _df[~_bad_mask].copy()

    # ── Step 2: re-derive country from address ───────────────────────────────
    _original = _df["country"].copy()

    def _reextract_country(row) -> str:
        addr  = str(row.get("address") or "")
        addr_lines = [ln.strip() for ln in addr.splitlines() if ln.strip()]
        for candidate in [
            addr_lines[-1] if addr_lines else "",
            "\n".join(addr_lines[-3:]),
            "\n".join(addr_lines),
            str(row.get("manufacturer_name") or ""),
        ]:
            c = extract_country(candidate)
            if c:
                return c
        return str(row.get("country") or "")

    _df["country"] = _df.apply(_reextract_country, axis=1)

    changed = _df["country"] != _original.reindex(_df.index).fillna("")
    n_changed = int(changed.sum())
    if n_changed:
        print(f"\n[FIX] {n_changed} country value(s) corrected:")
        _report = _df[changed][["medicine_name", "manufacturer_name", "country"]].copy()
        _report["was"] = _original.reindex(_df.index)[changed].values
        for _, r in _report.iterrows():
            med = str(r["medicine_name"] or "")[:30]
            mfr = str(r["manufacturer_name"] or "")[:40]
            print(f"  {med:30}  {mfr:40}  {r['was']!r:20} -> {r['country']!r}")
    else:
        print("[FIX] No country values changed.")

    # ── Step 3: save both files ──────────────────────────────────────────────
    _df.to_csv(_inc_path,   index=False)
    _df.to_csv(_final_path, index=False)
    print(f"\n[FIX] {n_before} rows → {len(_df)} rows ({n_dropped} bad rows removed)")
    print(f"[FIX] Saved to:\n       {_inc_path}\n       {_final_path}")
    print("\nContinue from Step 5 (HHI) — no need to re-run Steps 3 or 4.")


[FIX] Re-deriving country and dropping bad rows for ALL records …
[FIX] Dropping 576 bad row(s) (sentence-fragment / symbol manufacturer names):
  Startvac              'nan'
  Tadalafil Mylan       'responsible for the release of the concerned batch.'
  Fubelv                '(Biotech Division)'
  Crysvita              'allphamed PHARBIL Arzneimittel GmbH'
  Jemperli              'responsible for the release of the concerned batch.'
  Orphacol              'responsible for the release of the concerned batch.'
  Lopinavir/Ritonavir   'responsible for the release of the concerned batch.'
  Carvykti              'date:'
  Pomalidomide Teva     'responsible for the release of the concerned batch'
  Yesintek              'ustekinumab'
  Yesintek              '• If you have or have had a recent infection.'
  Yesintek              'will receive the next dose of 90 mg Yesintek after 8 weeks, '
  Yesintek              'weeks. Your doctor will decide when you should receive your '
  Yesintek   

In [38]:
# Step 4 — consolidate & save
print("[4/6] Writing extraction tables …")
manufacturer_df = (
    pd.read_csv(incremental_path) if incremental_path.exists()
    else pd.DataFrame(columns=incremental_cols)
)
manufacturer_df.to_csv(OUTPUT_DIR / "manufacturers_extracted.csv", index=False)
pd.DataFrame(failures).to_csv(OUTPUT_DIR / "failures.csv", index=False)
print(f"Total rows: {len(manufacturer_df)}")
manufacturer_df.head()

[4/6] Writing extraction tables …
Total rows: 3054


,medicine_name,active_substance,atc_code,medicine_url,pdf_url,source_block,manufacturer_name,address,country,manufacturer_step,latitude,longitude
0,Eydenzelt,aflibercept,S01LA05,https://www.ema.europa.eu/en/medicines/human/E...,https://www.ema.europa.eu/en/documents/product...,A. MANUFACTURER OF THE BIOLOGICAL ACTIVE SUBST...,CELLTRION INC.,23 Academy-Ro\nYeonsu-gu\n22014 Incheon\nRepub...,Republic of Korea,1.0,NaN,NaN
1,Eydenzelt,aflibercept,S01LA05,https://www.ema.europa.eu/en/medicines/human/E...,https://www.ema.europa.eu/en/documents/product...,A. MANUFACTURER OF THE BIOLOGICAL ACTIVE SUBST...,Nuvisan France SARL,"2400, Route des Colles,\n06410, Biot,\nFrance",France,2.0,NaN,NaN
2,Eydenzelt,aflibercept,S01LA05,https://www.ema.europa.eu/en/medicines/human/E...,https://www.ema.europa.eu/en/documents/product...,A. MANUFACTURER OF THE BIOLOGICAL ACTIVE SUBST...,Midas Pharma GmbH,"Rheinstr. 49,\n55218 Ingelheim,\nGermany\nKYMO...",Spain,2.0,NaN,NaN
3,Kefdensis,denosumab,M05BX04,https://www.ema.europa.eu/en/medicines/human/E...,https://www.ema.europa.eu/en/documents/product...,A. MANUFACTURER OF THE BIOLOGICAL ACTIVE SUBST...,Alvotech hf,Sæmundargata 15-19\n102 Reykjavik\nIceland,Iceland,1.0,NaN,NaN
4,Kefdensis,denosumab,M05BX04,https://www.ema.europa.eu/en/medicines/human/E...,https://www.ema.europa.eu/en/documents/product...,A. MANUFACTURER OF THE BIOLOGICAL ACTIVE SUBST...,STADA Arzneimittel AG,Stadastrasse 2–18\n61118 Bad Vilbel\nGermany,Germany,2.0,NaN,NaN


In [39]:
manufacturer_df[
    manufacturer_df["active_substance"].str.contains("arsenic trioxide", case=False, na=False)
][["medicine_name", "manufacturer_name", "address", "country", "manufacturer_step"]]

,medicine_name,manufacturer_name,address,country,manufacturer_step
770,Arsenic trioxide Accord,Accord Healthcare Polska Sp. z o.o.,ul. Lutomierska 50\n95-200 Pabianice\nPOLAND,Poland,2.0
1140,Trisenox,"Almac Pharma Services Limited,","Almac House,\n20 Seagoe Industrial Estate,\nCr...",United Kingdom,2.0
1141,Trisenox,Almac Pharma Services (Ireland) Limited,"Finnabair Industrial Estate,\nDundalk, Co. Lou...",Ireland,2.0
1142,Trisenox,Merckle GmbH,"Graf-Arco-Str-3,\n89079 Ulm,\nGermany",Germany,2.0
1143,Trisenox,S.C. Sindan-Pharma S.R.L.,"B-dul Ion Mihalache nr 11, sector 1,\nCod 0111...",Romania,2.0


In [40]:
def build_hhi_tables(manufacturer_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    usable = manufacturer_df.dropna(subset=["country"]).copy()
    
    # Define columns up front so empty returns still have a schema
    hhi_cols_substance = ["active_substance", "n_records", "n_countries", "top_country", "top_country_share", "hhi"]
    hhi_cols_atc       = ["atc_code",         "n_records", "n_countries", "top_country", "top_country_share", "hhi"]
    
    if usable.empty:
        return pd.DataFrame(columns=hhi_cols_substance), pd.DataFrame(columns=hhi_cols_atc)

    def _make_rows(groupby_col, group_key):
        rows = []
        for val, grp in usable.groupby(groupby_col, dropna=True):
            if pd.isna(val):
                continue
            vc = grp["country"].value_counts()
            rows.append({
                group_key:           val,
                "n_records":         len(grp),
                "n_countries":       grp["country"].nunique(dropna=True),
                "top_country":       vc.idxmax(),
                "top_country_share": vc.max() / vc.sum(),
                "hhi":               compute_hhi(grp["country"]),
            })
        return pd.DataFrame(rows).sort_values(["hhi", "n_records"], ascending=[False, False])

    return _make_rows("active_substance", "active_substance"), _make_rows("atc_code", "atc_code")

In [41]:
# Step 5 — HHI
print("[5/6] Computing HHI tables …")
usable_df = manufacturer_df.dropna(subset=["manufacturer_name", "country"], how="any").copy()

# Exclude step-2 (batch release) manufacturers only when the same medicine
# also has a step-1 (active substance) record — i.e. biologics where upstream
# and downstream sites are explicitly distinguished.
#
# Rationale: for standard small-molecule generics (India, China etc.) the SmPC
# has only ONE manufacturer section headed 'responsible for batch release', so
# the actual manufacturing site is also labeled step=2.  Excluding ALL step-2
# rows would erase those countries from the analysis.  We therefore only drop
# step-2 rows for medicines that have a separate step-1 counterpart.
medicines_with_step1 = set(
    usable_df.loc[usable_df["manufacturer_step"] == 1, "medicine_url"].dropna()
)
step2_to_drop = (
    (usable_df["manufacturer_step"] == 2) &
    (usable_df["medicine_url"].isin(medicines_with_step1))
)
n_excluded = step2_to_drop.sum()
usable_df = usable_df[~step2_to_drop].copy()
print(f"[INFO] Excluded {n_excluded} step-2 batch-release records "
      f"(biologics with explicit step-1 upstream manufacturer)")
print(f"[INFO] Remaining records for HHI: {len(usable_df)}")

by_substance, by_atc = build_hhi_tables(usable_df)
by_substance.to_csv(OUTPUT_DIR / "hhi_by_active_substance.csv", index=False)
by_atc.to_csv(OUTPUT_DIR / "hhi_by_atc.csv", index=False)

if not usable_df.empty:
    country_counts = (
        usable_df
        .groupby(["atc_code", "active_substance", "country"], dropna=True)
        .size().reset_index(name="n_records")
        .sort_values(["atc_code", "active_substance", "n_records"], ascending=[True, True, False])
    )
else:
    country_counts = pd.DataFrame(columns=["atc_code", "active_substance", "country", "n_records"])
country_counts.to_csv(OUTPUT_DIR / "country_counts.csv", index=False)

# Step 6 — summary
print("[6/6] Done!")
for f in sorted(OUTPUT_DIR.glob("*.csv")):
    try:
        rows = len(pd.read_csv(f))
    except pd.errors.EmptyDataError:
        rows = 0
    print(f"  {f.name:<52} {rows:>6} rows")


[5/6] Computing HHI tables …
[INFO] Excluded 572 step-2 batch-release records (biologics with explicit step-1 upstream manufacturer)
[INFO] Remaining records for HHI: 2436
[6/6] Done!
  country_counts.csv                                     1837 rows
  ema_medicines_filtered.csv                             1858 rows
  ema_medicines_raw.csv                                  2701 rows
  failures.csv                                              0 rows
  hhi_by_active_substance.csv                            1006 rows
  hhi_by_atc.csv                                          860 rows
  manufacturers_extracted.csv                            3054 rows
  manufacturers_extracted_incremental.csv                3054 rows


## 17. Explore Results

In [43]:
# Manufacturer records
manufacturer_df[["medicine_name", "manufacturer_name", "address", "country"]] \
    .dropna(subset=["manufacturer_name"]).head(20)

,medicine_name,manufacturer_name,address,country
0,Eydenzelt,CELLTRION INC.,23 Academy-Ro\nYeonsu-gu\n22014 Incheon\nRepub...,Republic of Korea
1,Eydenzelt,Nuvisan France SARL,"2400, Route des Colles,\n06410, Biot,\nFrance",France
2,Eydenzelt,Midas Pharma GmbH,"Rheinstr. 49,\n55218 Ingelheim,\nGermany\nKYMO...",Spain
3,Kefdensis,Alvotech hf,Sæmundargata 15-19\n102 Reykjavik\nIceland,Iceland
4,Kefdensis,STADA Arzneimittel AG,Stadastrasse 2–18\n61118 Bad Vilbel\nGermany,Germany
5,Kadcyla,Lonza Ltd.,Lonzastrasse\nCH-3930 Visp\nSwitzerland,Switzerland
6,Kadcyla,F. Hoffmann La Roche AG,Grenzacherstrasse 124\nCH-4058 Basel\nSwitzerland,Switzerland
7,Kadcyla,Roche Pharma AG,Emil-Barell-Strasse 1\n79639 Grenzach-Whylen\n...,Germany
8,Nucala,GlaxoSmithKline LLC,"893 River Road\nConshohocken,\nPA 19428\nUnite...",United States
9,Nucala,"Human Genome Sciences, Inc.","9911 Belward Campus Drive\nRockville, MD 20850...",United States


In [44]:
# Country distribution of manufacturing sites — step 1 (biological active substance) only
manufacturer_df[manufacturer_df["manufacturer_step"] == 1]["country"].value_counts().head(20)


country
United States        185
Germany               59
Republic of Korea     43
Ireland               36
Denmark               36
Switzerland           32
India                 32
China                 28
Austria               24
Singapore             21
Belgium               20
France                18
Spain                 14
Italy                 13
United Kingdom        12
Slovenia              11
Netherlands           11
Japan                 10
Iceland                8
Sweden                 7
Name: count, dtype: int64

In [45]:
# HHI by active substance (higher = more geographically concentrated supply chain)
by_substance.head(20) if not by_substance.empty else print("No HHI data yet.")

,active_substance,n_records,n_countries,top_country,top_country_share,hhi
730,pitolisant,4,1,France,1.0,1.0
841,sitagliptin,4,1,Netherlands,1.0,1.0
846,sitagliptin;metformin hydrochloride,4,1,Netherlands,1.0,1.0
74,Ibuprofen,3,1,France,1.0,1.0
105,Mepolizumab,3,1,United States,1.0,1.0
297,corifollitropin alfa,3,1,Netherlands,1.0,1.0
326,delafloxacin meglumine,3,1,Italy,1.0,1.0
401,epoetin alfa,3,1,Slovenia,1.0,1.0
435,filamentous haemagglutinin;Haemophilus influen...,3,1,France,1.0,1.0
485,hepatitis B surface antigen;Diphtheria toxoid;...,3,1,France,1.0,1.0


In [46]:
# HHI by ATC code
by_atc.head(20) if not by_atc.empty else print("No HHI data yet.")

,atc_code,n_records,n_countries,top_country,top_country_share,hhi
25,A10AB04;A10AD04,4,1,France,1.0,1.0
316,J05A,4,1,Ireland,1.0,1.0
750,N07XX11,4,1,France,1.0,1.0
0,A02BC02,3,1,Germany,1.0,1.0
173,C01EB16,3,1,France,1.0,1.0
238,G03GA09,3,1,Netherlands,1.0,1.0
297,J01MA23,3,1,Italy,1.0,1.0
380,J07BC20,3,1,Belgium,1.0,1.0
556,L01XX41,3,1,Germany,1.0,1.0
691,N02CD06,3,1,Ireland,1.0,1.0


## Debug: Test Extraction on a Single PDF

Change `_DEBUG_PDF_URL` and run to inspect what the parser extracts — useful for investigating edge cases.
Does **not** write anything to the CSV.


In [48]:
# ── Quick debug: parse a single PDF and print extracted manufacturers ──────────
# Change the URL below and run this cell to test extraction on any EMA PDF.
# Does NOT write to the CSV — read-only inspection only.

_DEBUG_PDF_URL = "https://www.ema.europa.eu/en/documents/product-information/lyrica-epar-product-information_en.pdf"

import io
if "session" not in dir():  # build session if Step 3 hasn't run yet
    session = build_session()
print(f"[DEBUG] Fetching: {_DEBUG_PDF_URL}")
_pdf_bytes = download_binary(session, _DEBUG_PDF_URL)
print(f"[DEBUG] Downloaded {len(_pdf_bytes):,} bytes")

_text = extract_pdf_text_with_fallback(_pdf_bytes)
_candidate_blocks = split_into_candidate_blocks(_text)
_relevant_blocks  = keep_relevant_blocks(_candidate_blocks)
print(f"[DEBUG] {len(_relevant_blocks)} relevant manufacturer block(s) found\n")

_step_labels = {1: "bio.active", 2: "batch release", None: ""}
_all_results = []

for _bi, _block in enumerate(_relevant_blocks):
    _parsed = parse_manufacturer_block_multi(_block)
    print(f"--- Block {_bi+1} (first line: {_block.splitlines()[0]!r}) ---")
    if not _parsed:
        print("  (no manufacturers parsed)")
    for _name, _addr, _country, _step in _parsed:
        _tag = f" [{_step_labels[_step]}]" if _step else ""
        print(f"  → {_name} | {_country}{_tag}")
        if _addr:
            for _ln in _addr.splitlines():
                print(f"       {_ln}")
    print()
    _all_results.extend(_parsed)

print(f"Total: {len(_all_results)} manufacturer record(s)")


[DEBUG] Fetching: https://www.ema.europa.eu/en/documents/product-information/lyrica-epar-product-information_en.pdf
[DEBUG] Downloaded 654,895 bytes
[INFO] PDF: 140 total pages, reading 140 (all pages)
[DEBUG] 2 relevant manufacturer block(s) found

--- Block 1 (first line: 'A. MANUFACTURER(S) RESPONSIBLE FOR BATCH RELEASE') ---
  (no manufacturers parsed)

--- Block 2 (first line: 'A. MANUFACTURER(S) RESPONSIBLE FOR BATCH RELEASE') ---
  → Pfizer Manufacturing Deutschland GmbH | Germany [batch release]
       Mooswaldallee 1
       79108 Freiburg Im Breisgau
       Germany
       or
  → Mylan Hungary Kft. | Hungary [batch release]
       Mylan utca 1
       Komárom, 2900
       Hungary
       or
  → MEDIS INTERNATIONAL a.s., výrobní závod Bolatice | Czech Republic [batch release]
       Průmyslová 961/16
       747 23 Bolatice
       Czech Republic
  → Viatris International Supply Point BV | Belgium [batch release]
       Terhulpsesteenweg 6A
       1560 Hoeilaart
       Belgium
     